# bordeus — ingestion e debug del retrieval

Notebook di lavoro per la pipeline di ingestion **semi-automatica**,
allineato ai moduli in `src/bordeus_ingest/` (sostituisce la versione
che pilotava il vecchio crawl HTML+PDF).

Due usi:

1. **Eseguire l'ingestion passo per passo**, guardando cosa succede a
   ogni stadio invece di lanciare `bordeus-ingest sync` e sperare.
2. **Debuggare il retrieval**: dato un oggetto che il bot non trova, in
   quale chunk sta la risposta, e in che posizione la restituisce la
   ricerca per similarità.

## Il flusso

    fonti del gestore (PDF, immagini)
        │  extract-vocabolario / extract-calendario   ← non in questo notebook
        ▼
    knowledge/<area>/**.md   (Markdown curato, riletto a mano)
        │  discover → split → embed
        ▼                                    │  parse
    vector store (una collection per area)   ▼
                                        raccolta_date (tabella)

Il calendario non passa dal vector store: le date vanno in Postgres e il
bot le legge con il tool calling. Qui si vedono entrambi i rami.

## Una nota sul chunking, che è dove nascono i problemi di retrieval

Le tabelle del vocabolario producono **un chunk per riga**. Non è un
dettaglio: raggruppare più voci per chunk rompe la ricerca per
similarità, perché l'embedding diventa la media di oggetti scorrelati.
Con ~7 voci per chunk il segnale di ciascuna vale un settimo, e una
domanda su una tazza da caffè finisce per inseguire la parola "caffè" in
"Cialda caffè" invece dell'oggetto. Lo Step 5 qui sotto serve a
verificarlo caso per caso.

In [ ]:
import sys

sys.path.insert(0, "../../common/src")
sys.path.insert(0, "../src")

import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

from bordeus_common.log import setup_logging
from bordeus_ingest import chunk as chunk_mod
from bordeus_ingest import documents, knowledge, pipeline
from bordeus_ingest.calendario import parse as parse_calendario

setup_logging("INFO")
pd.set_option("display.max_colwidth", None)

## Configurazione

`area` accetta un id (risolto sotto `knowledge/`) o un percorso a una
cartella d'area. Il `DATABASE_URL` arriva da `../.env`.

In [ ]:
import os

database_url = os.environ["DATABASE_URL"]
area_ref = "sub-ato-e"

area = pipeline.load_manifest(area_ref)

print(f"Area:     {area.id}  ({area.nome})")
print(f"Gestore:  {area.gestore}")
print(f"Cartella: {area.area_dir}")
print(f"\nComuni ({len(area.comuni)}):")
for c in area.comuni:
    mat = c.mappatura()
    print(f"  {c.id:10} {c.nome:12} materiali dichiarati: {len(mat)}")
print(f"\nFrazioni con schema proprio ({len(area.frazioni)}):")
for f in area.frazioni:
    print(f"  {f.comune_id}/{f.hamlet:12} {f.nome}")

## Step 1 — Scoperta dei documenti RAG

Il `tipo` di ogni documento è il nome della cartella che lo contiene
(`vocabolario/`, `info/`, ...). Le cartelle che iniziano con `_` sono di
servizio e non vengono mai ingerite come contenuto RAG: in particolare
`_calendari/`, che ha una destinazione diversa.

Se un tipo che ti aspetti non compare qui, il file è nel posto
sbagliato: nessun errore, semplicemente non verrà mai recuperato.

In [ ]:
docs = documents.discover(area.area_dir, area.id)

df_docs = pd.DataFrame([
    {
        "source": d.metadata["source"],
        "tipo": d.metadata["tipo"],
        "comune_id": d.metadata["comune_id"] or "(area)",
        "caratteri": len(d.page_content),
    }
    for d in docs
])

print(f"{len(docs)} documenti\n")
print(df_docs.groupby(["tipo", "comune_id"]).agg(file=("source", "count"), caratteri=("caratteri", "sum")))
df_docs.head(30)

## Step 2 — Chunking

**È lo stadio che determina la qualità del retrieval**, più del modello
di embedding e più del prompt.

Le tabelle diventano un chunk per riga, resa come frase autonoma
("Tazzina in ceramica — Conferimento: RUR"), con oggetto e categoria
anche nei metadata. La prosa resta divisa per intestazioni `#`/`##`, con
un fallback testuale per le sezioni oltre `chunk_size`.

Cosa guardare: la lunghezza mediana e il numero di chunk. Se il
vocabolario ha ~400 voci e ottieni ~60 chunk, le righe si stanno
raggruppando e il retrieval ne soffrirà.

In [ ]:
chunks = chunk_mod.split_documents(docs)

df_chunks = pd.DataFrame([
    {
        "source": c.metadata["source"],
        "tipo": c.metadata["tipo"],
        "oggetto": c.metadata.get("oggetto", ""),
        "conferimento": c.metadata.get("conferimento", ""),
        "caratteri": len(c.page_content),
        "testo": c.page_content,
    }
    for c in chunks
])

righe_tabella = sum(
    len([r for r in d.page_content.splitlines() if r.strip().startswith("|") and "---" not in r]) - 1
    for d in docs
    if "| ---" in d.page_content or "|---" in d.page_content
)

print(f"{len(docs)} documenti -> {len(chunks)} chunk")
print(f"righe di tabella nei sorgenti: {righe_tabella} (dovrebbero diventare altrettanti chunk)")
print(f"\ncaratteri per chunk: min={df_chunks.caratteri.min()} "
      f"mediana={int(df_chunks.caratteri.median())} max={df_chunks.caratteri.max()}")
print(f"chunk con un oggetto identificato: {(df_chunks.oggetto != '').sum()}")

df_chunks.sample(min(8, len(df_chunks)))[["oggetto", "conferimento", "testo"]]

### Cerca una voce nei chunk

Prima di dare la colpa al retrieval, verifica che la risposta esista.
Un oggetto assente dal vocabolario non è un problema di embedding: è un
problema di dati, e si corregge nel Markdown.

In [ ]:
def cerca_nei_chunk(termine: str) -> pd.DataFrame:
    """Ricerca testuale (non semantica) fra i chunk: c'è o non c'è."""
    m = df_chunks[df_chunks.testo.str.contains(termine, case=False, regex=False)]
    print(f"{len(m)} chunk contengono {termine!r}")
    return m[["source", "oggetto", "conferimento", "testo"]]


cerca_nei_chunk("ceramica")

## Step 3 — Calendari

Ramo separato: `_calendari/<comune>/<periodo>.md` e
`_calendari/<comune>/_frazioni/<hamlet>/<periodo>.md`. Il percorso dice
a chi si applica il file; il manifest dice solo quale flusso locale
raccoglie quale materiale.

Le categorie qui sotto devono combaciare con quelle dichiarate in
`area.toml`: `sync` fallisce elencando i nomi reali se non è così.

In [ ]:
trovati = knowledge.discover_calendari(area.area_dir)

righe_cal = []
for t in trovati:
    date = parse_calendario(t.path.read_text(encoding="utf-8"))
    righe_cal.append({
        "source": t.source,
        "destinazione": t.etichetta,
        "date": len(date),
        "categorie": ", ".join(sorted({r.categoria.upper() for r in date})) or "(file vuoto)",
    })

df_cal = pd.DataFrame(righe_cal)
print(f"{len(trovati)} file di calendario, {df_cal.date.sum()} date totali\n")

print("Schema di raccolta dichiarato per destinazione:")
for (comune_id, hamlet) in sorted({(t.comune_id, t.hamlet) for t in trovati}):
    mappa = area.mappatura_per(comune_id, hamlet)
    etichetta = f"{comune_id}/{hamlet}" if hamlet else comune_id
    carta = mappa.get("carta", "(non dichiarato)")
    cartone = mappa.get("cartone", "(non dichiarato)")
    nota = "  <- schema DIVISO" if carta != cartone else ""
    print(f"  {etichetta:22} carta={carta!r} cartone={cartone!r}{nota}")

df_cal

## Step 4 — Scrittura su Postgres

`run_area` fa tutto: anagrafica (comuni e frazioni), calendari in
`raccolta_date`, chunk nel vector store.

**`reset=True` quando cambia la strategia di chunking.** L'id di un
chunk include il suo contenuto, quindi l'upsert aggiorna quelli che
ricalcola identici ma **non cancella** quelli che l'ingestion non
produce più. Dopo un cambio di chunking, senza reset, i vecchi chunk
restano nella collection e continuano a competere nel retrieval — e una
correzione ai dati sembra non avere effetto.

Caricare il modello di embedding richiede qualche secondo e della VRAM.

In [ ]:
from bordeus_common.embed import get_embeddings

embeddings = get_embeddings()

vectorstore = pipeline.run_area(
    area,
    database_url,
    embeddings=embeddings,
    con_rag=True,
    con_calendari=True,
    reset=True,   # metti False per un aggiornamento incrementale
)
print("\nIngestion completata")

## Step 5 — Debug del retrieval

La parte per cui questo notebook esiste. Il retrieval del bot usa
**solo la descrizione dell'oggetto** come query (non la frase intera
dell'utente: saluti e formulazione sono rumore), con
`QUERY_INSTRUCTION` come prefisso — il modello di embedding è
decoder-only con last-token pooling e le query vanno istruite, i
documenti no.

`diagnosi()` risponde alla domanda che serve davvero: *la voce giusta
c'è, e a che posizione esce?*

- **assente dai chunk** → problema di dati, correggi il Markdown;
- **presente ma fuori dai primi k** → problema di retrieval: chunk troppo
  grossi, query poco specifica, o `k` troppo basso;
- **presente nei primi k ma la risposta è sbagliata** → problema di
  prompt, e si continua in `bot/notebooks/rag_eval.ipynb`.

In [ ]:
from bordeus_bot.rag import DEFAULT_K_VOCABOLARIO, QUERY_INSTRUCTION, vocabolario_filter
from bordeus_common.vectorstore import get_vectorstore

# Stesso modello dell'ingestion, ma con l'istruzione di query attiva:
# è la configurazione che usa il bot, e va replicata o i punteggi non
# sono confrontabili con quelli reali.
embeddings_query = get_embeddings(query_instruction=QUERY_INSTRUCTION)
vs = get_vectorstore(database_url, area.id, embeddings_query)

comune_id = "donnas"


def diagnosi(descrizione_oggetto: str, atteso: str | None = None, k: int = DEFAULT_K_VOCABOLARIO):
    """Mostra cosa recupera il bot per questa descrizione, e dove finisce
    la voce attesa. `atteso` è una sottostringa dell'oggetto giusto."""
    risultati = vs.similarity_search_with_score(
        descrizione_oggetto, k=k, filter=vocabolario_filter(comune_id)
    )
    print(f"query: {descrizione_oggetto!r}   (k={k}, comune={comune_id!r})\n")
    for i, (doc, distanza) in enumerate(risultati, start=1):
        marca = "  <<<" if atteso and atteso.lower() in doc.page_content.lower() else ""
        print(f"  {i:2}. dist={distanza:.4f}  {doc.page_content}{marca}")

    if atteso:
        trovato = any(atteso.lower() in d.page_content.lower() for d, _ in risultati)
        print(f"\n{atteso!r} nei primi {k}: {'SI' if trovato else 'NO'}")
        if not trovato:
            # Esiste nella collection ma non è stato recuperato? Allarghiamo.
            largo = vs.similarity_search_with_score(
                descrizione_oggetto, k=100, filter=vocabolario_filter(comune_id)
            )
            pos = next(
                (i for i, (d, _) in enumerate(largo, 1) if atteso.lower() in d.page_content.lower()),
                None,
            )
            if pos:
                print(f"-> esce in posizione {pos} su 100: problema di RANKING, non di dati.")
            else:
                print("-> non esce nemmeno nei primi 100: controlla che la voce esista")
                print("   (usa cerca_nei_chunk) e che il filtro per comune sia giusto.")
    return risultati


diagnosi("tazza per caffè, ceramica gialla con manico", atteso="ceramica")

### Altre query di prova

Aggiungine quante ne servono. Le più utili sono quelle che hanno dato
una risposta sbagliata in chat: qui si vede subito se il contesto
recuperato conteneva o no la voce giusta.

In [ ]:
prove = [
    ("bottiglia di plastica", "bottiglia"),
    ("scatola di cartone", "cartone"),
    ("giornale vecchio", "giornal"),
    ("pile esaurite", "batteria"),
    ("vecchio tostapane rotto", "tostapane"),
]

for descrizione, atteso in prove:
    diagnosi(descrizione, atteso=atteso, k=5)
    print("-" * 78)

## Step 6 — Visualizzazione degli embedding (opzionale)

Riduzione t-SNE degli embedding della collection: chunk semanticamente
vicini finiscono vicini nel grafico. Utile per vedere a colpo d'occhio
se il vocabolario si raggruppa per categoria di conferimento — se i
punti sono un'unica nuvola indistinta, il retrieval avrà poco da
discriminare.

Richiede le dipendenze del gruppo `dev` (`matplotlib`, `scikit-learn`),
import lazy dentro `viz.py`.

In [ ]:
from bordeus_ingest import viz

righe = viz.fetch_embeddings_for_collection(database_url, area.id)
print(f"{len(righe)} chunk nella collection {area.id!r}")

# Colora per categoria di conferimento quando c'è (il vocabolario la
# mette nei metadata), altrimenti per tipo: sono le due dimensioni per
# cui ha senso aspettarsi dei gruppi.
etichette = [
    r["metadata"].get("conferimento") or r["metadata"].get("tipo", "?")
    for r in righe
]
coords = viz.reduce([r["embedding"] for r in righe], n_components=2)
viz.plot(coords, etichette, title=f"Chunk di {area.id} (t-SNE)")

## Da qui in poi

- **La voce non c'è** → correggi il Markdown in `knowledge/<area>/`,
  poi rilancia lo Step 4 con `reset=True`.
- **Il ranking è sbagliato** → guarda la lunghezza dei chunk allo Step 2.
  Se le righe di tabella non sono una per chunk, è lì il problema.
- **Il contesto è giusto ma la risposta no** → è il prompt:
  `bot/notebooks/rag_eval.ipynb`, che confronta varianti di system prompt
  sullo stesso percorso reale (tool calling compreso).
- **Il giorno di raccolta è sbagliato** → non è RAG: è la mappatura dei
  materiali in `area.toml` e la tabella `raccolta_date`. Guarda lo
  Step 3 e i log TRACE del bot.